# 20 · Generative Models

- **Level:** Modern (2013 → present)
- **Typical semester:** Semester 7–8 electives and research
- **Prerequisites:** 05 (§9 SDEs), 11, 14, 16, 18
- **Matches code in:** `lipipy.dl` (base models) and `lipipy.research` (diffusion, flow matching)


## 1. The goal

Learn $p_\theta(x)\approx p_{data}(x)$ so we can sample and/or evaluate likelihood. Families: explicit tractable (autoregressive, flows), explicit approximate (VAE, diffusion), implicit (GAN), energy-based.


## 2. Autoregressive models

$p(x) = \prod_ip(x_i|x_{<i})$ (PixelCNN, WaveNet, GPT). Exact likelihood, sequential sampling.


## 3. Variational autoencoder (VAE)

**Evidence lower bound:**
$$\log p_\theta(x)\ge\mathcal L = \mathbb E_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x)\,\|\,p(z))$$
$$\log p_\theta(x) = \mathcal L + D_{KL}(q_\phi(z|x)\|p_\theta(z|x))$$

**Reparameterization:** $z = \mu_\phi(x) + \sigma_\phi(x)\odot\epsilon$, $\epsilon\sim\mathcal N(0,I)$

KL term for a diagonal Gaussian encoder: $\frac12\sum_j(\mu_j^2 + \sigma_j^2 - 1 - \log\sigma_j^2)$

**$\beta$-VAE:** weight the KL by $\beta$. **IWAE bound:** $\mathbb E\left[\log\frac1K\sum_k\frac{p(x, z_k)}{q(z_k|x)}\right]$

**VQ-VAE:** $z_q = e_{k^*}$, $k^* = \arg\min_k\|z_e - e_k\|$; loss $\log p(x|z_q) + \|\text{sg}[z_e] - e\|^2 + \beta\|z_e - \text{sg}[e]\|^2$ (sg = stop-gradient). **FSQ:** round each latent dimension to a few levels.


## 4. Generative adversarial networks

$$\min_G\max_DV(D,G) = \mathbb E_{x\sim p_{data}}[\log D(x)] + \mathbb E_{z\sim p(z)}[\log(1 - D(G(z)))]$$

Optimal discriminator $D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}$; then $V = -\log4 + 2\,JS(p_{data}\|p_g)$.

Non-saturating generator loss: $-\mathbb E[\log D(G(z))]$.

**WGAN:** $\min_G\max_{\|D\|_L\le1}\mathbb E[D(x)] - \mathbb E[D(G(z))]$; gradient penalty $\lambda\mathbb E_{\hat x}\left[(\|\nabla_{\hat x}D(\hat x)\|_2 - 1)^2\right]$

**Hinge GAN loss:** $L_D = \mathbb E[\max(0, 1 - D(x))] + \mathbb E[\max(0, 1 + D(G(z)))]$, $L_G = -\mathbb E[D(G(z))]$

Conditional GAN: $D(x|y)$, $G(z|y)$. **CycleGAN** cycle loss $\|F(G(x)) - x\|_1$.

Evaluation: **FID** $= \|\mu_r - \mu_g\|^2 + \text{tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r\Sigma_g)^{1/2}\right)$; **Inception score** $\exp\left(\mathbb E_xKL(p(y|x)\|p(y))\right)$


## 5. Normalizing flows

$x = f(z)$ invertible, $z\sim p_Z$:
$$\log p_X(x) = \log p_Z(f^{-1}(x)) + \log\left|\det\frac{\partial f^{-1}}{\partial x}\right|$$

**Affine coupling (RealNVP):** $y_{1:d} = x_{1:d}$, $\;y_{d+1:D} = x_{d+1:D}\odot\exp(s(x_{1:d})) + t(x_{1:d})$; $\;\log|\det J| = \sum s(x_{1:d})$

Glow adds invertible $1\times1$ convolutions ($\log|\det| = hw\log|\det W|$). Autoregressive flows (MAF, IAF).

**Continuous normalizing flow:** $\frac{dz}{dt} = f(z,t)$, $\;\frac{d\log p(z_t)}{dt} = -\text{tr}\left(\frac{\partial f}{\partial z}\right)$


## 6. Energy-based models

$p_\theta(x) = \frac{e^{-E_\theta(x)}}{Z_\theta}$, $Z_\theta = \int e^{-E_\theta(x)}dx$

Gradient: $\nabla_\theta\log p(x) = -\nabla_\theta E(x) + \mathbb E_{x'\sim p_\theta}[\nabla_\theta E(x')]$; samples via Langevin dynamics. Contrastive divergence (RBMs), noise-contrastive estimation.

**RBM:** $E(v,h) = -a^\top v - b^\top h - v^\top Wh$, $\;p(h_j = 1|v) = \sigma(b_j + v^\top W_{:j})$


## 7. Score matching

Score $s(x) = \nabla_x\log p(x)$ (does not need $Z$).

Explicit: $\frac12\mathbb E\|s_\theta(x) - \nabla\log p(x)\|^2$. Implicit (Hyvärinen): $\mathbb E\left[\text{tr}(\nabla_xs_\theta) + \frac12\|s_\theta\|^2\right]$.

**Denoising score matching:** with $\tilde x = x + \sigma\epsilon$, $\;\mathbb E\left\|s_\theta(\tilde x) + \frac{\epsilon}{\sigma}\right\|^2$, since $\nabla_{\tilde x}\log q(\tilde x|x) = -\frac{\tilde x - x}{\sigma^2}$

**Tweedie's formula:** $\mathbb E[x|\tilde x] = \tilde x + \sigma^2\nabla_{\tilde x}\log p(\tilde x)$


## 8. Denoising diffusion probabilistic models (DDPM)

**Forward process:** $q(x_t|x_{t-1}) = \mathcal N(\sqrt{1 - \beta_t}x_{t-1}, \beta_tI)$. With $\alpha_t = 1 - \beta_t$, $\bar\alpha_t = \prod_{s\le t}\alpha_s$:
$$q(x_t|x_0) = \mathcal N(\sqrt{\bar\alpha_t}x_0, (1 - \bar\alpha_t)I), \qquad x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\,\epsilon$$

**Posterior:** $q(x_{t-1}|x_t, x_0) = \mathcal N(\tilde\mu_t, \tilde\beta_tI)$, $\;\tilde\mu_t = \frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1 - \bar\alpha_t}x_0 + \frac{\sqrt{\alpha_t}(1 - \bar\alpha_{t-1})}{1 - \bar\alpha_t}x_t$, $\;\tilde\beta_t = \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\beta_t$

**Simple training loss:** $\mathcal L_{simple} = \mathbb E_{t, x_0, \epsilon}\left\|\epsilon - \epsilon_\theta(\sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\epsilon, t)\right\|^2$

**Sampling:** $x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\epsilon_\theta(x_t,t)\right) + \sigma_tz$

Link to score: $s_\theta(x_t, t) = -\frac{\epsilon_\theta(x_t, t)}{\sqrt{1 - \bar\alpha_t}}$

Noise schedules: linear $\beta_t$, cosine $\bar\alpha_t = \frac{f(t)}{f(0)}$, $f(t) = \cos^2\left(\frac{t/T + s}{1 + s}\cdot\frac\pi2\right)$. Signal-to-noise ratio $\text{SNR}(t) = \frac{\bar\alpha_t}{1 - \bar\alpha_t}$.

Parameterizations: $\epsilon$-prediction, $x_0$-prediction, $v$-prediction $v = \sqrt{\bar\alpha_t}\epsilon - \sqrt{1 - \bar\alpha_t}x_0$.


## 9. DDIM and fast samplers

**DDIM (deterministic, $\eta = 0$):**
$$x_{t-1} = \sqrt{\bar\alpha_{t-1}}\underbrace{\left(\frac{x_t - \sqrt{1 - \bar\alpha_t}\epsilon_\theta}{\sqrt{\bar\alpha_t}}\right)}_{\hat x_0} + \sqrt{1 - \bar\alpha_{t-1}}\,\epsilon_\theta$$

DPM-Solver, Heun (EDM), UniPC: higher-order ODE solvers on the probability-flow ODE.


## 10. Score SDE view

Forward SDE $dx = f(x,t)dt + g(t)dW$. Variance-preserving (VP): $dx = -\frac12\beta(t)x\,dt + \sqrt{\beta(t)}dW$. Variance-exploding (VE): $dx = \sqrt{\frac{d\sigma^2}{dt}}dW$.

**Reverse SDE:** $dx = [f - g^2\nabla_x\log p_t(x)]dt + g\,d\bar W$

**Probability-flow ODE** (same marginals, deterministic): $\frac{dx}{dt} = f - \frac12g^2\nabla_x\log p_t(x)$

**EDM (Karras et al.):** $x = x_0 + \sigma\epsilon$, denoiser $D_\theta(x;\sigma) = c_{skip}(\sigma)x + c_{out}(\sigma)F_\theta(c_{in}(\sigma)x; c_{noise}(\sigma))$, with $c_{skip} = \frac{\sigma_d^2}{\sigma^2 + \sigma_d^2}$, $c_{out} = \frac{\sigma\sigma_d}{\sqrt{\sigma^2 + \sigma_d^2}}$, $c_{in} = \frac{1}{\sqrt{\sigma^2 + \sigma_d^2}}$


## 11. Guidance

**Classifier guidance:** $\nabla_x\log p(x|y) = \nabla_x\log p(x) + w\nabla_x\log p(y|x)$

**Classifier-free guidance:** $\tilde\epsilon = (1 + w)\epsilon_\theta(x_t, c) - w\,\epsilon_\theta(x_t, \varnothing)$ (often written $\epsilon_\varnothing + s(\epsilon_c - \epsilon_\varnothing)$); trained by randomly dropping the condition

**Latent diffusion:** run diffusion on $z = \mathcal E(x)$ from a pretrained autoencoder. **DiT:** a transformer denoiser on latent patches, conditioned with adaptive layer norm (adaLN).


## 12. Flow matching and rectified flow

Interpolate $x_t = (1 - t)x_0 + tx_1$, $x_0\sim\mathcal N(0,I)$, $x_1\sim p_{data}$; target velocity $u_t = x_1 - x_0$.

**Conditional flow-matching loss:** $\mathcal L_{CFM} = \mathbb E_{t, x_0, x_1}\|v_\theta(x_t, t) - (x_1 - x_0)\|^2$

Sample by integrating $\frac{dx}{dt} = v_\theta(x, t)$ from $t = 0$ to $1$. **Rectified flow / reflow:** retrain on the model's own $(x_0, x_1)$ pairs to straighten trajectories. General Gaussian paths $x_t = \alpha_tx_1 + \sigma_tx_0$ recover diffusion as a special case.


## 13. Consistency models and distillation

Consistency function $f_\theta(x_t, t)\approx x_0$ for every point on a probability-flow trajectory; $f_\theta(x, \epsilon) = x$ at the boundary.

Consistency training loss: $d\left(f_\theta(x_{t_{n+1}}, t_{n+1}),\, f_{\theta^-}(x_{t_n}, t_n)\right)$ with $\theta^-$ an EMA / stop-gradient copy. Enables one- or few-step sampling.

Progressive distillation halves the number of sampling steps per round. Adversarial and distribution-matching distillation (DMD) also give few-step models.


## 14. Discrete and masked diffusion for text and tokens

Masked (absorbing-state) diffusion: each token is replaced by [MASK] with probability $1 - \alpha_t$; the loss is a weighted masked-LM cross-entropy
$$\mathcal L = \mathbb E_t\left[\frac{-\alpha_t'}{1 - \alpha_t}\sum_{i:\,x_t^i = \text{[M]}}-\log p_\theta(x_0^i|x_t)\right]$$
Used by diffusion language models and MaskGIT-style image generators.
